# 🔍 Deblur/Unblur Image - Training on Google Colab
**Đồ án TGMT - 3 cấp độ: Face, Scene, ID Card**

Notebook này sẽ:
1. Clone mã nguồn dự án từ GitHub của bạn
2. Mount Google Drive để lưu model an toàn
3. Tải dataset (ví dụ: GoPro cho Scene)
4. Train model SwinIR
5. Lưu kết quả về Google Drive

## 1️⃣ Kiểm tra GPU & Mount Google Drive
Việc Mount Drive là BẮT BUỘC để lưu trữ model checkpoint. Colab sẽ tự xóa dữ liệu khi bạn tắt tab.

In [ ]:
!nvidia-smi
print('---')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Tải Mã Nguồn Từ GitHub
Chúng ta sẽ kéo (clone) code trực tiếp từ kho GitHub của bạn. Bất cứ khi nào bạn sửa code trên máy tính và push lên Github, bạn chỉ cần chạy lại cell này để lấy code mới nhất.

In [ ]:
import os

# Luôn quay về thư mục gốc của Colab trước khi xử lý
%cd /content

# Xóa thư mục cũ (nếu bạn chạy lại cell này để cập nhật code)
!rm -rf ThiGiacMayTinh

# Clone code từ GitHub của bạn
!git clone https://github.com/hnihTyoB/ThiGiacMayTinh.git

# Di chuyển vào thư mục dự án
%cd ThiGiacMayTinh

# QUAN TRỌNG: Gỡ bỏ bản basicsr lỗi thời nếu có trên Colab
!pip uninstall -y basicsr

# Cài đặt các thư viện cần thiết
!pip install -q lmdb pyyaml tb-nightly tqdm gdown

print('\n[✓] Dự án đã sẵn sàng!')

## 3️⃣ Tải Dataset
### Scene: GoPro Large (~9GB)

In [ ]:
# TỐI ƯU HÓA: Copy dataset zip từ Drive vào Colab để giải nén cực nhanh
# (Bỏ qua tải mạng)
!mkdir -p /content/ThiGiacMayTinh/downloads
!cp /content/drive/MyDrive/GOPRO_Large.zip /content/ThiGiacMayTinh/downloads/

# Chạy script tự động giải nén và sắp xếp dataset
!python scripts/data_preparation/download_deblur_datasets.py --scene

### Face & ID Card (Tuỳ chọn)
Nếu bạn train Face hoặc ID Card, hãy upload ảnh gốc (sharp) lên thư mục Drive của bạn, sau đó copy vào môi trường Colab và chạy script tạo blur.

In [ ]:
# 1. Tạo thư mục đích
!mkdir -p datasets/face/train/sharp

# 2. Copy zip từ Drive vào Colab (rất nhanh)
!cp /content/drive/MyDrive/celeba_hq_256.zip /content/

# 3. Giải nén vào thư mục sharp
!unzip -q /content/celeba_hq_256.zip -d datasets/face/train/sharp/

# 4. Dùng lệnh 'mv' di chuyển toàn bộ ảnh từ thư mục con ra đúng vị trí
!mv datasets/face/train/sharp/celeba_hq_256/* datasets/face/train/sharp/
# 5. Dọn dẹp file zip và thư mục nháp cho nhẹ ổ cứng Colab
!rm /content/celeba_hq_256.zip
print("[✓] Đã chuyển toàn bộ ảnh sharp vào đúng thư mục: datasets/face/train/sharp/")

# 4. Chạy lệnh tạo blur:
!python scripts/data_preparation/create_blur_dataset.py --input datasets/face/train/sharp/ --output datasets/face/train/blur --task face

## 4️⃣ Training
Bỏ comment (`#`) ở lệnh bạn muốn chạy. Mặc định đang train Scene.

In [ ]:
# ========== TẠO SYMLINK ĐỂ TỰ ĐỘNG LƯU VÀO DRIVE ==========
!mkdir -p /content/drive/MyDrive/DeblurModels
!rm -rf /content/ThiGiacMayTinh/experiments
!ln -s /content/drive/MyDrive/DeblurModels /content/ThiGiacMayTinh/experiments

# ========== TRAIN SCENE DEBLUR ==========
!python basicsr/train.py -opt options/train/train_deblur_scene.yml

# ========== TRAIN FACE DEBLUR ==========
# !python basicsr/train.py -opt options/train/train_deblur_face.yml

# ========== TRAIN ID CARD DEBLUR ==========
# !python basicsr/train.py -opt options/train/train_deblur_idcard.yml

## 5️⃣ Test & Inference (Tùy chọn)
Kiểm tra thử độ hiệu quả trực tiếp trên Colab.

In [ ]:
import os
os.makedirs('test_images', exist_ok=True)
os.makedirs('results', exist_ok=True)

TASK = 'scene'
MODEL_PATH = 'experiments/DeblurScene_SwinIR/models/net_g_40000.pth'

# Upload 1-2 ảnh mờ vào thư mục test_images/ bên thanh bên trái trước khi chạy lệnh này
if os.path.exists(MODEL_PATH):
    !python inference/inference_deblur.py \
        --input test_images \
        --output results \
        --model_path {MODEL_PATH} \
        --task {TASK}
else:
    print(f'⚠️ Chưa có model tại: {MODEL_PATH}. Hãy train trước!')